# 04 · Findings from code review and reproduction

The three most valuable outputs of this validation did not come from a statistic; they came from
reading the deployed code and reproducing it. Each finding is reproduced here with data.

| # | model | severity | finding |
|---|---|---|---|
| F1 | GBM (deployed) | High | Scoring by a **single Monte-Carlo path** adds noise of order σ√T and destroys the ranking |
| F2 | GBM (any) | Medium | The expected score is a monotone transform of trailing mean log return — the "simulation" adds nothing |
| F3 | CLAM | Critical | Training sequences are built from 94 tickers concatenated and sorted **by date**, so a 252-row training window spans ~2.7 days of 94 stocks while inference feeds 252 days of one stock |
| F4 | CLAM | Blocking | Deployed weights were trained through 2025-08; only ~13 months of out-of-time data exist, ~5 independent 65-day horizons |
| F5 | Momentum | High | Entering one trading day after the rebalance removes roughly half of the CAGR — the excess return is concentrated in the first session, so the backtest is only valid under close-of-rebalance execution |

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import config
from sv import db

con = db.connect(read_only=True)
q   = lambda name, **p: db.run_sql_file(con, name, p or None)   # run sql/queries/<name>.sql
sql = lambda text, **p: db.read(con, text, p or None)                # run an inline query
pd.set_option("display.width", 140); plt.rcParams["figure.figsize"] = (10, 4)

## F1 · Single-path GBM vs its expectation

In [ ]:
q("champion_challenger", start=config.BACKTEST_START).set_index("strategy").loc[["gbm", "gbm_expected", "momentum", "universe_ew"]].round(3)

In [ ]:
sql("""SELECT strategy, ROUND(AVG(turnover), 3) AS avg_weekly_turnover FROM portfolio_returns
       WHERE strategy IN ('gbm', 'gbm_expected', 'momentum') GROUP BY strategy ORDER BY strategy""")

In [ ]:
# rank correlation between the single-draw score and its expectation, per week: how much of the ranking survives the draw
s = sql("""SELECT a.date, a.ticker, a.score AS mc, b.score AS expected
           FROM signals a JOIN signals b USING (date, ticker)
           WHERE a.model = 'gbm' AND b.model = 'gbm_expected'""")
rho = s.groupby("date").apply(lambda g: g[["mc", "expected"]].corr(method="spearman").iloc[0, 1])
print(f"median weekly Spearman(mc, expected) = {rho.median():.3f}")
ax = rho.plot(title="Weekly rank correlation: single MC path vs expectation"); ax.set_ylim(0, 1); plt.show()

## F2 · The expectation is just trailing mean return

`E[score] = mean_i exp(mu * t_i) - 1` depends on μ only, and exp is monotone, so ranking by the expected
GBM score is ranking by 2-year mean log return. Verified below: rank correlation ≈ 1.

In [ ]:
from sv import features
adj = features.load_wide(con, "adj_close")
dates = features.rebalance_dates(con)
mu = np.log(adj).diff().rolling(config.GBM_LOOKBACK, min_periods=int(config.GBM_LOOKBACK * .9)).mean().reindex(dates)
mu = mu.stack(future_stack=True).rename("mu").reset_index(); mu.columns = ["date", "ticker", "mu"]
e = sql("SELECT date, ticker, score FROM signals WHERE model = 'gbm_expected'").merge(mu, on=["date", "ticker"])
print("Spearman(expected score, trailing mean log return) =", round(e[["score", "mu"]].corr(method="spearman").iloc[0, 1], 4))

## F3 · CLAM training / inference mismatch (code review)

From `Quant_Model_Research/clam_model.py` (paraphrased): the training set concatenates 94 tickers'
daily rows, sorts by date, then slides a 252-row window over the result. With 94 tickers per date,
252 consecutive rows cover **2.7 calendar days**, mixing 94 different stocks. `clam_inference.py`
feeds **252 days of a single stock**. The network was therefore never trained on the input it
receives in production. No retraining can validate the deployed weights; the recommendation is
redevelopment with per-ticker windows.

Evidence that the mismatch matters is in `02_monitoring` once CLAM scores are loaded: `psi_score`
of `clam_orig` on the 3,000-stock universe versus its 94 large-cap development set.

## F4 · In-sample contamination of the deployed CLAM weights

Weights dated 2025-08-22 → every backtest week before that date is in-sample. Out-of-time: 2025-08-22 → 2026-09-14.

In [ ]:
sql("""SELECT COUNT(DISTINCT date) AS oot_weeks, COUNT(DISTINCT date) / 13.0 AS independent_65d_horizons
       FROM panel WHERE date > $t""", t=config.CLAM_ORIGINAL_TRAIN_END)

## F5 · Execution-timing sensitivity of momentum

`ret_fwd_1w_lag1` enters at the close of the *next* trading day instead of the rebalance close.
The difference is the return of a single session per week.

In [ ]:
q("champion_challenger", start=config.OOT_START).set_index("strategy").loc[["momentum", "momentum_lag1", "gbm_expected", "gbm_expected_lag1"]].round(3)

In [ ]:
sql("""
WITH r AS (SELECT p.date, p.ret_fwd_1w, p.ret_fwd_1w_lag1,
                  RANK() OVER (PARTITION BY p.date ORDER BY s.score DESC) AS rk
           FROM panel p JOIN signals s USING (date, ticker)
           WHERE s.model = 'momentum' AND p.tradable)
SELECT EXTRACT(year FROM date) AS year,
       ROUND(AVG(CASE WHEN rk <= 50 THEN ret_fwd_1w - ret_fwd_1w_lag1 END) * 52, 3) AS top50_first_session_ann,
       ROUND(AVG(CASE WHEN rk <= 50 THEN ret_fwd_1w_lag1 END) * 52, 3)              AS top50_rest_of_week_ann,
       ROUND(AVG(ret_fwd_1w - ret_fwd_1w_lag1) * 52, 3)                             AS universe_first_session_ann
FROM r GROUP BY 1 ORDER BY 1""")

The first session after each rebalance contributes as much as the other four combined for the
top-50 momentum names, far above the universe average. Whatever the cause (short-horizon continuation,
Friday-close microstructure), a user who cannot trade at the rebalance close does not get the
backtested return. This becomes a prohibited-use condition.

## Survivorship bias, quantified

See `00_data_quality`: the number of today's tickers with prices rises from ~1,400 (2015) to ~2,600 (2026).
The bias inflates every candidate's absolute return in the same direction; it does not favour one
candidate over another, which is why all pass/fail decisions are made on *active* return vs the
same universe.